# SAM2 AUTO BOX VS AGRICLIP
"" POINT PROMPTS

In [37]:
# ============================================================
# CELL 1 — IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms

from datasets import load_dataset

print("=" * 70)
print("AGRICLIP + SAM2 PROMPT STRATEGY NOTEBOOK")
print("=" * 70)

print("PyTorch:", torch.__version__)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)

AGRICLIP + SAM2 PROMPT STRATEGY NOTEBOOK
PyTorch: 2.12.0+cu130
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [38]:
# ============================================================
# CELL 2 — PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/home/zahranadeem/thesis"
)

AGRICLIP_ROOT = (
    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
)

AGRICLIP_ALIGNMENT_DIR = (
    AGRICLIP_ROOT
    / "AgriCLIP_alignment"
)

# ------------------------------------------------------------
# DINO
# ------------------------------------------------------------

DINO_CANDIDATES = [
    PROJECT_ROOT
    / "agriclip_Zahra"
    / "weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
    / "Weights"
    / "DINO-PT.pth",

    PROJECT_ROOT
    / "DINO-PT.pth"
]

DINO_PATH = None

for candidate in DINO_CANDIDATES:
    if candidate.exists():
        DINO_PATH = candidate
        break

# ------------------------------------------------------------
# AgriCLIP aligner
# ------------------------------------------------------------

ALIGNER_PATH = (
    PROJECT_ROOT
    / "vegann_aligner.pth"
)

# ------------------------------------------------------------
# Original non-augmented fine-tuned SAM2
# ------------------------------------------------------------

SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = (
    "facebook/sam2.1-hiera-tiny"
)

print("=" * 70)
print("PATH CHECK")
print("=" * 70)

print("Project:")
print(PROJECT_ROOT)

print("\nAgriCLIP:")
print(AGRICLIP_ROOT)

print("\nDINO:")
print(DINO_PATH)

print("\nAligner:")
print(ALIGNER_PATH)

print("\nSAM2:")
print(SAM2_CHECKPOINT)

print("\nSAM2 exists:",
      SAM2_CHECKPOINT.exists())

print("Aligner exists:",
      ALIGNER_PATH.exists())

print("DINO exists:",
      DINO_PATH is not None and DINO_PATH.exists())

if DINO_PATH is None:
    raise FileNotFoundError(
        "DINO-PT.pth could not be found."
    )

assert ALIGNER_PATH.exists()
assert SAM2_CHECKPOINT.exists()

print("\n✓ All required weight files found")

PATH CHECK
Project:
/home/zahranadeem/thesis

AgriCLIP:
/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP

DINO:
/home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth

Aligner:
/home/zahranadeem/thesis/vegann_aligner.pth

SAM2:
/home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt

SAM2 exists: True
Aligner exists: True
DINO exists: True

✓ All required weight files found


In [39]:
# ============================================================
# CELL 3 — AGRICLIP PYTHON PATH
# ============================================================

if str(AGRICLIP_ALIGNMENT_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(AGRICLIP_ALIGNMENT_DIR)
    )

print("=" * 70)
print("AGRICLIP MODULE PATH")
print("=" * 70)

print(
    AGRICLIP_ALIGNMENT_DIR
)

print(
    "\n✓ AgriCLIP alignment directory added to sys.path"
)

AGRICLIP MODULE PATH
/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP/AgriCLIP_alignment

✓ AgriCLIP alignment directory added to sys.path


In [40]:
# ============================================================
# CELL 4 — IMPORT TEXTTOCONCEPT
# ============================================================

from TextToConcept import (
    TextToConcept,
    ClipZeroShot
)

print("=" * 70)
print("TEXTTOCONCEPT")
print("=" * 70)

print(
    "TextToConcept:",
    TextToConcept
)

print(
    "ClipZeroShot:",
    ClipZeroShot
)

print("=" * 70)

TEXTTOCONCEPT
TextToConcept: <class 'TextToConcept.TextToConcept'>
ClipZeroShot: <class 'TextToConcept.ClipZeroShot'>


In [41]:
# ============================================================
# CELL 5 — LOAD VEGANN
# ============================================================

print("=" * 70)
print("LOADING VEGANN")
print("=" * 70)

vegann = load_dataset(
    "simonMadec/VegAnn"
)

print(vegann)

print(
    "\nOriginal dataset size:",
    len(vegann["train"])
)

LOADING VEGANN
DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5'],
        num_rows: 3775
    })
})

Original dataset size: 3775


In [42]:
# ============================================================
# CELL 6 — CREATE EXACT VEGANN TRAIN / VALIDATION SPLIT
# ============================================================

vegann_split = (
    vegann["train"]
    .train_test_split(
        test_size=0.20,
        seed=42
    )
)

train_hf = (
    vegann_split["train"]
)

val_hf = (
    vegann_split["test"]
)

print("=" * 70)
print("VEGANN SPLIT")
print("=" * 70)

print(
    "Training:",
    len(train_hf)
)

print(
    "Validation:",
    len(val_hf)
)

assert len(val_hf) == 755

print(
    "\n✓ Validation split = 755 images"
)

VEGANN SPLIT
Training: 3020
Validation: 755

✓ Validation split = 755 images


In [44]:
# ============================================================
# CELL 7 — VEGANN IMAGE / MASK KEYS
# ============================================================

print("=" * 70)
print("VEGANN COLUMNS")
print("=" * 70)

print(
    "Columns:",
    train_hf.column_names
)

# VegAnn normally uses image/mask.
# Detect automatically so the notebook does not depend
# on a stale variable from another notebook.

IMAGE_KEY = None
MASK_KEY = None

for key in [
    "image",
    "images",
    "img"
]:

    if key in train_hf.column_names:

        IMAGE_KEY = key
        break


for key in [
    "mask",
    "masks",
    "label",
    "labels",
    "segmentation",
    "annotation"
]:

    if key in train_hf.column_names:

        MASK_KEY = key
        break


print(
    "\nIMAGE_KEY:",
    IMAGE_KEY
)

print(
    "MASK_KEY:",
    MASK_KEY
)

if IMAGE_KEY is None:
    raise RuntimeError(
        "Could not find VegAnn image column."
    )

if MASK_KEY is None:
    raise RuntimeError(
        "Could not find VegAnn mask column."
    )

print(
    "\n✓ Dataset keys detected"
)

VEGANN COLUMNS
Columns: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5']

IMAGE_KEY: image
MASK_KEY: mask

✓ Dataset keys detected


In [45]:
# ============================================================
# CELL 8 — IMAGE + MASK HELPERS
# ============================================================

def get_image(
    index,
    split=val_hf
):
    """
    Return a VegAnn image as RGB PIL.Image.
    """

    image = split[index][IMAGE_KEY]

    if isinstance(
        image,
        Image.Image
    ):

        return image.convert("RGB")

    image = np.asarray(
        image
    )

    return Image.fromarray(
        image
    ).convert("RGB")


def get_mask(
    index,
    split=val_hf
):
    """
    Return VegAnn vegetation mask as uint8 binary array.
    """

    mask = split[index][MASK_KEY]

    if isinstance(
        mask,
        Image.Image
    ):

        mask = np.asarray(mask)

    elif torch.is_tensor(mask):

        mask = (
            mask.detach()
            .cpu()
            .numpy()
        )

    else:

        mask = np.asarray(mask)

    if mask.ndim == 3:

        mask = mask[..., 0]

    return (
        mask > 0
    ).astype(np.uint8)


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_image = get_image(0)
test_mask = get_mask(0)

print("=" * 70)
print("IMAGE / MASK TEST")
print("=" * 70)

print(
    "Image type:",
    type(test_image)
)

print(
    "Image size:",
    test_image.size
)

print(
    "Mask shape:",
    test_mask.shape
)

print(
    "Foreground:",
    int(test_mask.sum())
)

print("=" * 70)

IMAGE / MASK TEST
Image type: <class 'PIL.Image.Image'>
Image size: (512, 512)
Mask shape: (512, 512)
Foreground: 208628


In [46]:
# ============================================================
# CELL 9 — LOAD DINO RESNET-50
# ============================================================

print("=" * 70)
print("LOADING DINO RESNET-50")
print("=" * 70)

dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True
)

print(
    "DINO model:",
    type(dino_model)
)

print(
    "\nLoading checkpoint:",
    DINO_PATH
)

dino_state_dict = torch.load(
    DINO_PATH,
    map_location="cpu",
    weights_only=True
)

load_result = dino_model.load_state_dict(
    dino_state_dict,
    strict=True
)

print(
    "\nDINO load result:",
    load_result
)

dino_model = (
    dino_model
    .to(device)
)

dino_model.eval()

for parameter in dino_model.parameters():
    parameter.requires_grad = False

print(
    "\n✓ DINO loaded and frozen"
)

print(
    "Layer4:",
    dino_model.layer4
)

print("=" * 70)

LOADING DINO RESNET-50
DINO model: <class 'torchvision.models.resnet.ResNet'>

Loading checkpoint: /home/zahranadeem/thesis/agriclip_Zahra/weights/DINO-PT.pth


Using cache found in /home/zahranadeem/.cache/torch/hub/facebookresearch_dino_main
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zahranadeem/thesis/agriclip_sam2/agriclip_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



DINO load result: <All keys matched successfully>

✓ DINO loaded and frozen
Layer4: Sequential(
  (0): Bottleneck(
    (conv1): Conv2d(1024, 512, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv3): Conv2d(512, 2048, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn3): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(1024, 2048, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): Bottleneck(
    (conv1): Conv2d(2048, 512, kernel_size=(1, 1), str

In [49]:
# ============================================================
# CELL 10 — DINO PREPROCESSING
# ============================================================

dino_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print("=" * 70)
print("DINO TRANSFORM")
print("=" * 70)

print(dino_transform)

print("=" * 70)

DINO TRANSFORM
Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [50]:
# ============================================================
# CELL 11 — DINO LAYER4 HOOK
# ============================================================

captured_feature_map = None


def capture_layer4_output(
    module,
    inputs,
    output
):

    global captured_feature_map

    captured_feature_map = (
        output.detach()
    )


hook_handle = (
    dino_model.layer4.register_forward_hook(
        capture_layer4_output
    )
)

print("=" * 70)
print("DINO LAYER4 HOOK")
print("=" * 70)

print(
    "✓ Hook registered successfully"
)

print("=" * 70)

DINO LAYER4 HOOK
✓ Hook registered successfully


In [51]:
# ============================================================
# CELL 12 — DINO SPATIAL PATCH EXTRACTION
# ============================================================

@torch.no_grad()
def extract_dino_patch_features(
    pil_image
):

    global captured_feature_map

    captured_feature_map = None

    image_tensor = (
        dino_transform(
            pil_image
        )
        .unsqueeze(0)
        .to(device)
    )

    _ = dino_model(
        image_tensor
    )

    if captured_feature_map is None:

        raise RuntimeError(
            "DINO layer4 hook did not capture output."
        )

    if captured_feature_map.ndim != 4:

        raise RuntimeError(
            "Expected [B,C,H,W], got "
            f"{captured_feature_map.shape}"
        )

    batch_size, channels, grid_h, grid_w = (
        captured_feature_map.shape
    )

    patch_features = (
        captured_feature_map[0]
        .permute(
            1,
            2,
            0
        )
        .reshape(
            grid_h * grid_w,
            channels
        )
        .contiguous()
    )

    return (
        patch_features,
        grid_h,
        grid_w
    )


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_features, grid_h, grid_w = (
    extract_dino_patch_features(
        test_image
    )
)

print("=" * 70)
print("DINO PATCH TEST")
print("=" * 70)

print(
    "Raw feature map:",
    captured_feature_map.shape
)

print(
    "Grid:",
    f"{grid_h} × {grid_w}"
)

print(
    "Patch features:",
    test_features.shape
)

assert test_features.shape == (
    49,
    2048
)

print(
    "\n✓ DINO extraction gives [49, 2048]"
)

print("=" * 70)

DINO PATCH TEST
Raw feature map: torch.Size([1, 2048, 7, 7])
Grid: 7 × 7
Patch features: torch.Size([49, 2048])

✓ DINO extraction gives [49, 2048]


In [52]:
# ============================================================
# CELL 13 — LOAD VEGANN AGRICLIP ALIGNER
# ============================================================

print("=" * 70)
print("LOADING VEGANN ALIGNER")
print("=" * 70)

aligner_checkpoint = torch.load(
    ALIGNER_PATH,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(aligner_checkpoint.keys())
)

W = torch.as_tensor(
    aligner_checkpoint["W"],
    dtype=torch.float32
)

b = torch.as_tensor(
    aligner_checkpoint["b"],
    dtype=torch.float32
)

print(
    "W:",
    W.shape
)

print(
    "b:",
    b.shape
)

# ------------------------------------------------------------
# Normalize orientation
# ------------------------------------------------------------

if W.shape == (
    512,
    2048
):

    print(
        "Detected W = [512, 2048]"
    )

elif W.shape == (
    2048,
    512
):

    W = W.T

    print(
        "Detected W = [2048, 512]; transposed."
    )

else:

    raise RuntimeError(
        f"Unexpected aligner W shape: {W.shape}"
    )

b = b.reshape(-1)

assert W.shape == (
    512,
    2048
)

assert b.shape == (
    512,
)

vegann_aligner = nn.Linear(
    2048,
    512,
    bias=True
)

with torch.no_grad():

    vegann_aligner.weight.copy_(W)
    vegann_aligner.bias.copy_(b)

vegann_aligner = (
    vegann_aligner
    .to(device)
)

vegann_aligner.eval()

for parameter in (
    vegann_aligner.parameters()
):

    parameter.requires_grad = False

print(
    "\n✓ VegAnn aligner loaded"
)

print(
    "Weight:",
    vegann_aligner.weight.shape
)

print(
    "Bias:",
    vegann_aligner.bias.shape
)

print("=" * 70)

LOADING VEGANN ALIGNER
Checkpoint keys: ['b', 'W']
W: torch.Size([512, 2048])
b: torch.Size([512])
Detected W = [512, 2048]

✓ VegAnn aligner loaded
Weight: torch.Size([512, 2048])
Bias: torch.Size([512])


In [53]:
# ============================================================
# CELL 14 — CREATE TEXTTOCONCEPT
# ============================================================

print("=" * 70)
print("CREATING TEXTTOCONCEPT")
print("=" * 70)

text_to_concept = TextToConcept(
    dino_model,
    "Dino"
)

print(
    "TextToConcept:",
    type(text_to_concept)
)

print(
    "Internal CLIP:",
    type(text_to_concept.clip_model)
)

print(
    "\n✓ TextToConcept created"
)

print("=" * 70)

CREATING TEXTTOCONCEPT
TextToConcept: <class 'TextToConcept.TextToConcept'>
Internal CLIP: <class 'TextToConcept.ClipZeroShot'>

✓ TextToConcept created


In [54]:
# ============================================================
# CELL 15 — LOAD ALIGNER INTO TEXTTOCONCEPT
# ============================================================

text_to_concept.load_linear_aligner(
    str(ALIGNER_PATH)
)

print("=" * 70)
print("TEXTTOCONCEPT ALIGNER")
print("=" * 70)

print(
    "✓ Linear aligner loaded into TextToConcept"
)

print("=" * 70)

TEXTTOCONCEPT ALIGNER
✓ Linear aligner loaded into TextToConcept


In [56]:
# ============================================================
# CELL 16 — VEGETATION PROMPTS
# ============================================================

POSITIVE_PROMPTS = [

    "vegetation",
    "green vegetation",
    "crop plants",
    "crop vegetation",
    "agricultural plants",
    "green plants",
]

NEGATIVE_PROMPTS = [

    "soil",
    "bare soil",
    "ground",
    "bare ground",
    "non-vegetated ground",
    "agricultural soil",
]

print("Prompt strategy: POSITIVE / NEGATIVE")

print("\nPositive prompts:")
for i, prompt in enumerate(POSITIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")

print("\nNegative prompts:")
for i, prompt in enumerate(NEGATIVE_PROMPTS, 1):
    print(f"{i}. {prompt}")
    
    
    


Prompt strategy: POSITIVE / NEGATIVE

Positive prompts:
1. vegetation
2. green vegetation
3. crop plants
4. crop vegetation
5. agricultural plants
6. green plants

Negative prompts:
1. soil
2. bare soil
3. ground
4. bare ground
5. non-vegetated ground
6. agricultural soil


In [57]:

# ============================================================
# CELL 17 ENCODE POSITIVE AND NEGATIVE PROMPT SETS
# ============================================================

@torch.no_grad()
def encode_prompt_set(prompts):

    embeddings = []

    for prompt in prompts:

        embedding = text_to_concept.encode_text(
            [[prompt]]
        )

        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(
                embedding
            )

        if not torch.is_tensor(embedding):
            raise TypeError(
                "encode_text returned an unsupported type: "
                f"{type(embedding)}"
            )

        embedding = (
            embedding
            .detach()
            .float()
            .to(device)
            .reshape(-1)
        )

        embeddings.append(
            embedding
        )

    embeddings = torch.stack(
        embeddings,
        dim=0
    )

    # Normalize each prompt independently
    embeddings = F.normalize(
        embeddings,
        dim=1
    )

    # Average prompts
    mean_embedding = embeddings.mean(
        dim=0
    )

    # Normalize the averaged embedding
    mean_embedding = F.normalize(
        mean_embedding,
        dim=0
    )

    return embeddings, mean_embedding


# ------------------------------------------------------------
# POSITIVE
# ------------------------------------------------------------

positive_text_embeddings, positive_embedding = (
    encode_prompt_set(
        POSITIVE_PROMPTS
    )
)


# ------------------------------------------------------------
# NEGATIVE
# ------------------------------------------------------------

negative_text_embeddings, negative_embedding = (
    encode_prompt_set(
        NEGATIVE_PROMPTS
    )
)


print(
    "Positive embeddings:",
    positive_text_embeddings.shape
)

print(
    "Negative embeddings:",
    negative_text_embeddings.shape
)

print(
    "Positive mean embedding:",
    positive_embedding.shape
)

print(
    "Negative mean embedding:",
    negative_embedding.shape
)

Positive embeddings: torch.Size([6, 512])
Negative embeddings: torch.Size([6, 512])
Positive mean embedding: torch.Size([512])
Negative mean embedding: torch.Size([512])


In [58]:
# ============================================================
# POSITIVE - NEGATIVE CONCEPT
# ============================================================

vegetation_embedding = (
    positive_embedding
    - negative_embedding
)

vegetation_embedding = F.normalize(
    vegetation_embedding,
    dim=0
)

print(
    "Final vegetation embedding:",
    vegetation_embedding.shape
)

Final vegetation embedding: torch.Size([512])


In [79]:
# ============================================================
# CELL — DEFINE NON-EMPTY VEGANN EVALUATION SET
# ============================================================

NONEMPTY_INDICES = []
EMPTY_INDICES = []

for idx in range(len(val_hf)):

    gt_mask = get_mask(
        idx
    )

    if gt_mask.sum() == 0:

        EMPTY_INDICES.append(
            idx
        )

    else:

        NONEMPTY_INDICES.append(
            idx
        )

print("=" * 70)
print("VEGANN EVALUATION SET")
print("=" * 70)

print(
    "Total validation images:",
    len(val_hf)
)

print(
    "Non-empty GT images:",
    len(NONEMPTY_INDICES)
)

print(
    "Empty GT images:",
    len(EMPTY_INDICES)
)

print(
    "Percentage empty:",
    f"{100 * len(EMPTY_INDICES) / len(val_hf):.2f}%"
)

print(
    "\nEmpty image IDs:"
)

print(
    EMPTY_INDICES
)

assert len(EMPTY_INDICES) == 9
assert len(NONEMPTY_INDICES) == 746

print("\n✓ Evaluation sets created")

VEGANN EVALUATION SET
Total validation images: 755
Non-empty GT images: 746
Empty GT images: 9
Percentage empty: 1.19%

Empty image IDs:
[81, 281, 348, 429, 454, 497, 498, 552, 644]

✓ Evaluation sets created


In [59]:
# ============================================================
# CELL 18 — TEST DINO → ALIGNER → TEXT SIMILARITY
# ============================================================

with torch.no_grad():

    aligned_test = (
        vegann_aligner(
            test_features.to(device)
        )
    )

    aligned_test = F.normalize(
        aligned_test,
        dim=-1
    )

    test_scores = (
        aligned_test
        @ vegetation_embedding
    )

print("=" * 70)
print("AGRICLIP SPATIAL SCORING")
print("=" * 70)

print(
    "DINO features:",
    test_features.shape
)

print(
    "Aligned features:",
    aligned_test.shape
)

print(
    "Vegetation embedding:",
    vegetation_embedding.shape
)

print(
    "Patch scores:",
    test_scores.shape
)

print(
    "\nScore min:",
    test_scores.min().item()
)

print(
    "Score max:",
    test_scores.max().item()
)

top_preview = torch.topk(
    test_scores,
    k=10
)

print(
    "\nTop-10 patch indices:",
    top_preview.indices.cpu().numpy()
)

print(
    "Top-10 scores:",
    top_preview.values.cpu().numpy()
)

print("=" * 70)

AGRICLIP SPATIAL SCORING
DINO features: torch.Size([49, 2048])
Aligned features: torch.Size([49, 512])
Vegetation embedding: torch.Size([512])
Patch scores: torch.Size([49])

Score min: -0.020329272374510765
Score max: 0.14176683127880096

Top-10 patch indices: [13 43  6 20 29 10  1 47 18  8]
Top-10 scores: [0.14176683 0.1323612  0.13027082 0.12215064 0.12186894 0.12176584
 0.1201911  0.1149954  0.11485691 0.11447184]


In [60]:
# ============================================================
# CELL 19 — AGRICLIP PATCHES → BOX + POINTS
# ============================================================

def agriclip_patches_to_box(
    image,
    top_indices,
    grid_h,
    grid_w,
    margin=25
):

    W, H = image.size

    patch_h = H / grid_h
    patch_w = W / grid_w

    patch_boxes = []

    for idx in top_indices:

        row = (
            int(idx) //
            grid_w
        )

        col = (
            int(idx) %
            grid_w
        )

        x1 = int(
            col * patch_w
        )

        y1 = int(
            row * patch_h
        )

        x2 = int(
            (col + 1) * patch_w
        )

        y2 = int(
            (row + 1) * patch_h
        )

        patch_boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )

    patch_boxes = np.asarray(
        patch_boxes,
        dtype=np.float32
    )

    xmin = patch_boxes[:, 0].min()
    ymin = patch_boxes[:, 1].min()
    xmax = patch_boxes[:, 2].max()
    ymax = patch_boxes[:, 3].max()

    xmin = max(
        0,
        xmin - margin
    )

    ymin = max(
        0,
        ymin - margin
    )

    xmax = min(
        W - 1,
        xmax + margin
    )

    ymax = min(
        H - 1,
        ymax + margin
    )

    return np.array(
        [
            xmin,
            ymin,
            xmax,
            ymax
        ],
        dtype=np.float32
    )


def agriclip_patches_to_points(
    image,
    top_indices,
    grid_h,
    grid_w
):

    W, H = image.size

    patch_h = H / grid_h
    patch_w = W / grid_w

    points = []

    for idx in top_indices:

        row = (
            int(idx) //
            grid_w
        )

        col = (
            int(idx) %
            grid_w
        )

        x1 = col * patch_w
        y1 = row * patch_h

        x2 = (
            col + 1
        ) * patch_w

        y2 = (
            row + 1
        ) * patch_h

        points.append(
            [
                (x1 + x2) / 2,
                (y1 + y2) / 2
            ]
        )

    return np.asarray(
        points,
        dtype=np.float32
    )


print(
    "✓ Box and point conversion functions ready"
)

✓ Box and point conversion functions ready


In [61]:
# ============================================================
# CELL 20 — LOAD ORIGINAL FINE-TUNED SAM2
# ============================================================

from transformers import (
    Sam2Model,
    Sam2Processor
)

print("=" * 70)
print("LOADING ORIGINAL FINE-TUNED SAM2")
print("=" * 70)

sam2_processor = (
    Sam2Processor.from_pretrained(
        MODEL_NAME
    )
)

sam2_model = (
    Sam2Model.from_pretrained(
        MODEL_NAME
    )
)

sam2_checkpoint = torch.load(
    SAM2_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint keys:",
    list(sam2_checkpoint.keys())
)

if (
    "mask_decoder_state_dict"
    not in sam2_checkpoint
):

    raise RuntimeError(
        "Expected mask_decoder_state_dict "
        "in SAM2 checkpoint."
    )

sam2_model.mask_decoder.load_state_dict(
    sam2_checkpoint[
        "mask_decoder_state_dict"
    ],
    strict=True
)

sam2_model = (
    sam2_model.to(device)
)

sam2_model.eval()

print(
    "\n✓ Fine-tuned non-augmented SAM2 loaded"
)

print(
    "Checkpoint epoch:",
    sam2_checkpoint.get("epoch")
)

print(
    "Best stored IoU:",
    sam2_checkpoint.get(
        "best_mean_iou"
    )
)

print("=" * 70)

LOADING ORIGINAL FINE-TUNED SAM2


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Checkpoint keys: ['epoch', 'model_name', 'mask_decoder_state_dict', 'optimizer_state_dict', 'best_mean_iou', 'history']

✓ Fine-tuned non-augmented SAM2 loaded
Checkpoint epoch: 8
Best stored IoU: 0.8724805513351916


In [62]:
# ============================================================
# CELL 21 — SAM2 BOX PREDICTION
# ============================================================

@torch.no_grad()
def sam2_box_prediction(
    image,
    box,
    threshold=0.5
):

    inputs = sam2_processor(
        images=image,
        input_boxes=[
            [
                box.tolist()
            ]
        ],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities": probabilities.cpu(),
        "scores": scores.cpu()
    }


print(
    "✓ SAM2 box prediction ready"
)

✓ SAM2 box prediction ready


In [63]:
# ============================================================
# CELL 22 — SAM2 POINT PREDICTION
# ============================================================

@torch.no_grad()
def sam2_point_prediction(
    image,
    points,
    threshold=0.5
):

    points = np.asarray(
        points,
        dtype=np.float32
    )

    if points.ndim != 2:
        raise ValueError(
            f"Expected [N,2], "
            f"got {points.shape}"
        )

    input_points = [
        [
            points.tolist()
        ]
    ]

    input_labels = [
        [
            [1] * len(points)
        ]
    ]

    inputs = sam2_processor(
        images=image,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities":
            probabilities.cpu(),
        "scores":
            scores.cpu()
    }


print(
    "✓ SAM2 point prediction ready"
)

✓ SAM2 point prediction ready


In [64]:
# ============================================================
# CELL 23 — SAM2 BOX + POSITIVE POINTS
# ============================================================

@torch.no_grad()
def sam2_box_point_prediction(
    image,
    box,
    points,
    threshold=0.5
):

    points = np.asarray(
        points,
        dtype=np.float32
    )

    if points.ndim != 2:
        raise ValueError(
            f"Expected points [N,2], "
            f"got {points.shape}"
        )

    input_boxes = [
        [
            box.tolist()
        ]
    ]

    input_points = [
        [
            points.tolist()
        ]
    ]

    input_labels = [
        [
            [1] * len(points)
        ]
    ]

    inputs = sam2_processor(
        images=image,
        input_boxes=input_boxes,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = sam2_model(
        **inputs
    )

    logits = outputs.pred_masks[
        0, 0
    ]

    scores = outputs.iou_scores[
        0, 0
    ]

    probabilities = torch.sigmoid(
        logits
    )

    height = int(
        inputs[
            "original_sizes"
        ][0, 0].item()
    )

    width = int(
        inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probabilities = F.interpolate(
        probabilities.unsqueeze(1),
        size=(
            height,
            width
        ),
        mode="bilinear",
        align_corners=False
    ).squeeze(1)

    candidate = (
        scores.argmax().item()
    )

    prediction = (
        probabilities[candidate]
        > threshold
    ).cpu().numpy().astype(
        np.uint8
    )

    return {
        "prediction": prediction,
        "confidence": float(
            scores[candidate].item()
        ),
        "candidate": candidate,
        "probabilities":
            probabilities.cpu(),
        "scores":
            scores.cpu()
    }


print(
    "✓ SAM2 box + points prediction ready"
)

✓ SAM2 box + points prediction ready


In [80]:
# ============================================================
# CELL 24 — METRICS
# ============================================================

# ============================================================
# CELL — ROBUST SEGMENTATION METRICS
# ============================================================

def compute_metrics_robust(
    prediction,
    ground_truth
):
    """
    Compute segmentation metrics.

    For the main VegAnn segmentation evaluation, empty
    ground-truth images should normally be excluded before
    calculating the dataset mean.

    This function also handles the empty/empty case safely.
    """

    prediction = (
        prediction > 0
    ).astype(bool)

    ground_truth = (
        ground_truth > 0
    ).astype(bool)

    pred_area = prediction.sum()
    gt_area = ground_truth.sum()

    intersection = np.logical_and(
        prediction,
        ground_truth
    ).sum()

    union = np.logical_or(
        prediction,
        ground_truth
    ).sum()

    # --------------------------------------------------------
    # Empty GT
    # --------------------------------------------------------

    if gt_area == 0:

        # Completely empty prediction
        if pred_area == 0:

            return {
                "iou": 1.0,
                "dice": 1.0,
                "precision": 1.0,
                "recall": 1.0,
                "empty_gt": True,
                "empty_prediction": True
            }

        # False positive prediction on empty image
        else:

            return {
                "iou": 0.0,
                "dice": 0.0,
                "precision": 0.0,
                "recall": 0.0,
                "empty_gt": True,
                "empty_prediction": False
            }

    # --------------------------------------------------------
    # Normal non-empty GT case
    # --------------------------------------------------------

    iou = (
        intersection / union
        if union > 0
        else 0.0
    )

    dice_den = (
        pred_area +
        gt_area
    )

    dice = (
        2.0 * intersection / dice_den
        if dice_den > 0
        else 0.0
    )

    precision = (
        intersection / pred_area
        if pred_area > 0
        else 0.0
    )

    recall = (
        intersection / gt_area
    )

    return {
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall),
        "empty_gt": False,
        "empty_prediction": (
            pred_area == 0
        )
    }


print("✓ Robust metric function ready.")

✓ Robust metric function ready.


In [66]:
# ============================================================
# CELL 25 — GT MASK → GT BOX
# ============================================================

def mask_to_box(
    mask
):

    ys, xs = np.where(
        mask > 0
    )

    if len(xs) == 0:

        raise ValueError(
            "Ground-truth mask is empty."
        )

    return np.array(
        [
            xs.min(),
            ys.min(),
            xs.max(),
            ys.max()
        ],
        dtype=np.float32
    )


def box_iou(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(
        ax1,
        bx1
    )

    iy1 = max(
        ay1,
        by1
    )

    ix2 = min(
        ax2,
        bx2
    )

    iy2 = min(
        ay2,
        by2
    )

    iw = max(
        0,
        ix2 - ix1
    )

    ih = max(
        0,
        iy2 - iy1
    )

    intersection = (
        iw * ih
    )

    area_a = (
        max(
            0,
            ax2 - ax1
        )
        *
        max(
            0,
            ay2 - ay1
        )
    )

    area_b = (
        max(
            0,
            bx2 - bx1
        )
        *
        max(
            0,
            by2 - by1
        )
    )

    union = (
        area_a +
        area_b -
        intersection
    )

    if union == 0:
        return 0.0

    return (
        intersection /
        union
    )


print(
    "✓ GT box helpers ready"
)

✓ GT box helpers ready



## Point 2 — Does semantic localization add value?

This focused experiment compares the same fine-tuned SAM2 checkpoint on the same non-empty VegAnn validation images using:
1. AgriCLIP semantic boxes with K=32 and K=49.
2. A full-image box.
3. A random-patch box made from 32 randomly selected DINO grid patches.
4. A nonsemantic random box matched to the area of that image's AgriCLIP K=32 box.

Random controls are deterministic (fixed seed plus image index). All methods use the same SAM2 prediction function, threshold, and evaluation images. The notebook also reports how often the K=32 semantic box covers at least 95% of the image and stratifies performance by GT vegetation coverage.


In [ ]:
# Define the localization helper explicitly.
@torch.no_grad()
def get_agriclip_localization(image, k=32):
    patch_features, grid_h, grid_w = extract_dino_patch_features(image)
    aligned_features = vegann_aligner(patch_features.to(device))
    aligned_features = F.normalize(aligned_features, dim=-1)
    scores = aligned_features @ vegetation_embedding
    k = min(int(k), len(scores))
    topk = torch.topk(scores, k=k)
    return {
        "top_indices": topk.indices.detach().cpu().numpy(),
        "top_scores": topk.values.detach().cpu().numpy(),
        "grid_h": int(grid_h),
        "grid_w": int(grid_w),
        "all_scores": scores.detach().cpu().numpy()
    }

print("Localization helper ready.")

In [ ]:
# Reproducibility and experiment settings
EXPERIMENT_SEED = 2026
RANDOM_PATCH_K = 32
WHOLE_IMAGE_COVERAGE_THRESHOLD = 0.95  # box area / image area
SPARSE_GT_COVERAGE_THRESHOLD = 0.05    # <=5% foreground pixels

rng_global = np.random.default_rng(EXPERIMENT_SEED)

print("Evaluation images:", len(NONEMPTY_INDICES))
print("K values:", [32, 49])
print("Random-patch count:", RANDOM_PATCH_K)
print("Whole-image criterion: box area >= 95% of image area")
print("Sparse-target criterion: GT foreground coverage <= 5%")

In [ ]:
def clip_box(box, width, height):
    """Clip [x1,y1,x2,y2] to image boundaries and ensure positive extent."""
    x1, y1, x2, y2 = map(float, box)
    x1 = float(np.clip(x1, 0, width - 1))
    y1 = float(np.clip(y1, 0, height - 1))
    x2 = float(np.clip(x2, x1 + 1, width))
    y2 = float(np.clip(y2, y1 + 1, height))
    return np.array([x1, y1, x2, y2], dtype=np.float32)

def random_patch_box(image, grid_h, grid_w, k=32, seed=0, margin=25):
    """Bounding rectangle around k randomly selected grid patches."""
    W, H = image.size
    rng = np.random.default_rng(seed)
    n = grid_h * grid_w
    chosen = rng.choice(n, size=min(k, n), replace=False)
    return agriclip_patches_to_box(image, chosen, grid_h, grid_w, margin=margin)

def area_matched_random_box(reference_box, width, height, seed=0):
    """
    Randomly located rectangle with approximately the same area and aspect ratio
    as the reference AgriCLIP box. It is not selected using image semantics.
    """
    x1, y1, x2, y2 = map(float, reference_box)
    target_area = max(1.0, (x2 - x1) * (y2 - y1))
    aspect = max(1e-6, (x2 - x1) / max(1e-6, y2 - y1))

    box_w = np.sqrt(target_area * aspect)
    box_h = target_area / box_w

    # Fit the rectangle inside the image while preserving its aspect ratio.
    scale = min(1.0, (width - 1) / box_w, (height - 1) / box_h)
    box_w *= scale
    box_h *= scale

    rng = np.random.default_rng(seed)
    x1r = rng.uniform(0, max(0, width - box_w))
    y1r = rng.uniform(0, max(0, height - box_h))
    return np.array([x1r, y1r, x1r + box_w, y1r + box_h], dtype=np.float32)

def box_area_fraction(box, width, height):
    x1, y1, x2, y2 = box
    return float(max(0, x2-x1) * max(0, y2-y1) / (width * height))

def evaluate_box_prompt(image, gt_mask, box):
    pred_out = sam2_box_prediction(image, box, threshold=0.5)
    metrics = compute_metrics_robust(pred_out["prediction"], gt_mask)
    return pred_out, metrics

print("Control-box and metric helpers ready.")

In [ ]:
# Main paired evaluation: all five prompt conditions on identical images.
# This performs 5 SAM2 inferences per non-empty validation image.
point2_rows = []

for idx in tqdm(NONEMPTY_INDICES, desc="Point 2: prompt ablations"):
    image = get_image(idx)
    gt_mask = get_mask(idx)
    W, H = image.size
    gt_coverage = float(gt_mask.sum() / gt_mask.size)

    # Extract DINO features and semantic scores once; reuse ranking for K=32 and K=49.
    loc49 = get_agriclip_localization(image, k=49)
    top32_indices = loc49["top_indices"][:32]
    box32 = agriclip_patches_to_box(
        image, top32_indices, loc49["grid_h"], loc49["grid_w"], margin=25
    )
    box49 = agriclip_patches_to_box(
        image, loc49["top_indices"], loc49["grid_h"], loc49["grid_w"], margin=25
    )

    # Random-patch box: same number of patches as K=32, no semantic ranking
    rand_patch = random_patch_box(
        image, loc49["grid_h"], loc49["grid_w"],
        k=RANDOM_PATCH_K, seed=EXPERIMENT_SEED + int(idx), margin=25
    )

    # Nonsemantic box matched to the area/aspect ratio of the semantic K=32 box
    area_matched = area_matched_random_box(
        box32, W, H, seed=EXPERIMENT_SEED + 100000 + int(idx)
    )

    # Full-image box
    full_image = np.array([0, 0, W - 1, H - 1], dtype=np.float32)

    prompts = {
        "AgriCLIP_K32": box32,
        "AgriCLIP_K49": box49,
        "Full_image": full_image,
        "Random_patch_K32": rand_patch,
        "Area_matched_random": area_matched,
    }

    for method, box in prompts.items():
        _, m = evaluate_box_prompt(image, gt_mask, box)
        point2_rows.append({
            "image_id": int(idx),
            "method": method,
            "iou": m["iou"],
            "dice": m["dice"],
            "precision": m["precision"],
            "recall": m["recall"],
            "gt_vegetation_coverage": gt_coverage,
            "box_area_fraction": box_area_fraction(box, W, H),
            "box_covers_at_least_95pct": (
                box_area_fraction(box, W, H) >= WHOLE_IMAGE_COVERAGE_THRESHOLD
            ),
            "box_x1": float(box[0]), "box_y1": float(box[1]),
            "box_x2": float(box[2]), "box_y2": float(box[3]),
        })

point2_df = pd.DataFrame(point2_rows)
print("Rows:", len(point2_df))
display(point2_df.head())

In [ ]:
# Aggregate comparison table
point2_summary = (
    point2_df.groupby("method")
    .agg(
        images=("image_id", "nunique"),
        mean_iou=("iou", "mean"),
        std_iou=("iou", "std"),
        mean_dice=("dice", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_box_area_pct=("box_area_fraction", lambda x: 100*x.mean()),
        whole_image_box_count=("box_covers_at_least_95pct", "sum"),
    )
    .reset_index()
)
point2_summary["mean_iou_pct"] = 100 * point2_summary["mean_iou"]
point2_summary["std_iou_pp"] = 100 * point2_summary["std_iou"]
point2_summary["whole_image_box_pct"] = (
    100 * point2_summary["whole_image_box_count"] / len(NONEMPTY_INDICES)
)
display(point2_summary.round(3))

In [ ]:
# How often does the semantic K=32 box already cover almost the whole image?
semantic32 = point2_df[point2_df["method"] == "AgriCLIP_K32"].copy()
whole_count = int(semantic32["box_covers_at_least_95pct"].sum())
print(f"K=32 boxes covering >=95% of image area: {whole_count}/{len(semantic32)} "
      f"({100*whole_count/len(semantic32):.2f}%)")
print(f"Mean K=32 box area: {100*semantic32['box_area_fraction'].mean():.2f}% of image")
print(f"Median K=32 box area: {100*semantic32['box_area_fraction'].median():.2f}% of image")

In [ ]:
# Stratify by GT vegetation coverage.
# Quantile groups give low/medium/high coverage using this evaluation population.
coverage_base = semantic32[["image_id", "gt_vegetation_coverage"]].drop_duplicates()
coverage_base["coverage_group"] = pd.qcut(
    coverage_base["gt_vegetation_coverage"],
    q=3,
    labels=["Low coverage", "Medium coverage", "High coverage"],
    duplicates="drop"
)

coverage_lookup = coverage_base[["image_id", "coverage_group"]]
stratified = point2_df.merge(coverage_lookup, on="image_id", how="left")

coverage_summary = (
    stratified.groupby(["coverage_group", "method"], observed=False)
    .agg(
        images=("image_id", "nunique"),
        mean_iou=("iou", "mean"),
        std_iou=("iou", "std"),
        mean_gt_coverage=("gt_vegetation_coverage", "mean"),
    )
    .reset_index()
)
coverage_summary["mean_iou_pct"] = 100 * coverage_summary["mean_iou"]
coverage_summary["std_iou_pp"] = 100 * coverage_summary["std_iou"]
coverage_summary["mean_gt_coverage_pct"] = 100 * coverage_summary["mean_gt_coverage"]
display(coverage_summary.round(3))

In [ ]:
# Explicit sparse-target subset (GT vegetation <=5% of image).
sparse_ids = set(
    coverage_base.loc[
        coverage_base["gt_vegetation_coverage"] <= SPARSE_GT_COVERAGE_THRESHOLD,
        "image_id"
    ]
)
sparse_summary = (
    point2_df[point2_df["image_id"].isin(sparse_ids)]
    .groupby("method")
    .agg(
        images=("image_id", "nunique"),
        mean_iou=("iou", "mean"),
        std_iou=("iou", "std"),
        mean_gt_coverage=("gt_vegetation_coverage", "mean")
    )
    .reset_index()
)
sparse_summary["mean_iou_pct"] = 100*sparse_summary["mean_iou"]
sparse_summary["std_iou_pp"] = 100*sparse_summary["std_iou"]
sparse_summary["mean_gt_coverage_pct"] = 100*sparse_summary["mean_gt_coverage"]
print(f"Sparse subset: GT vegetation coverage <=5%; n={len(sparse_ids)} images")
display(sparse_summary.round(3))

In [ ]:
# Save per-image results and summary tables.
POINT2_RESULTS_DIR = PROJECT_ROOT / "results" / "point2_semantic_localization"
POINT2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

point2_df.to_csv(POINT2_RESULTS_DIR / "point2_per_image_results.csv", index=False)
point2_summary.to_csv(POINT2_RESULTS_DIR / "point2_prompt_summary.csv", index=False)
coverage_summary.to_csv(POINT2_RESULTS_DIR / "point2_coverage_stratification.csv", index=False)
sparse_summary.to_csv(POINT2_RESULTS_DIR / "point2_sparse_target_summary.csv", index=False)

print("Saved Point 2 results to:", POINT2_RESULTS_DIR)